<a href="https://colab.research.google.com/github/antoine-bourgois/MIA_2026_UE16_NLP/blob/main/code/2_Cooccurrence_Tokens_Vecteurs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## TP 2 : Représentation Tokens - Matrice de cooccurrence

- Charger un Dataset
- Description statistique du dataset
- Analyse tokens vs lemmes
- Distribution des tokens

### Installation des librairies Python utiles pour le TP

In [1]:
!pip install tqdm jupyter ipywidgets datasets pyarrow huggingface_hub pandas matplotlib numpy scipy sklearn --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from tqdm.auto import tqdm
from collections import Counter

### Charger un jeu de données

In [3]:
from datasets import load_dataset_builder

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

builder = load_dataset_builder(dataset_path)

builder = load_dataset_builder(dataset_path, )

print(f"Le jeu de données {dataset_path!r} contient {len(builder.info.splits)} sous-ensembles :")
for name, split in builder.info.splits.items():
    print(f"    {name!r:<20}{split.num_examples:<6,} documents")

Le jeu de données 'AntoineBourgois/Gallica_Hors-d-Usage_cleaned' contient 5 sous-ensembles :
    'tous'              6,156  documents
    'auteur_genre'      200    documents
    'auteur'            200    documents
    'date_publication'  200    documents
    'genre_litteraire'  200    documents


In [4]:
# sous_ensemble, label_column = "auteur_genre", "auteur_genre"
# sous_ensemble, label_column = "auteur", "auteur"
sous_ensemble, label_column = "date_publication", "date_publication"
# sous_ensemble, label_column = "genre_litteraire", "genre_litteraire"

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

fichier_local = hf_hub_download(
    repo_id=dataset_path,
    filename=f"data/{sous_ensemble}-00000-of-00001.parquet",
    repo_type="dataset",
    force_download=True,
)

table = pq.read_table(fichier_local)
dataset = table.to_pylist()

print("Nombre de documents :", len(dataset))


def conversions_string_vers_liste(string):
    if isinstance(string, str):
        return string.split(" ")


for document in dataset:
    document["tokens"] = conversions_string_vers_liste(document["tokens"])
    document["lemmes"] = conversions_string_vers_liste(document["lemmes"])

data/date_publication-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  178MB            

data/date_publication-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Nombre de documents : 200


### Structure du dataset

In [6]:
print(f"Type dataset: {type(dataset)}")
print(f"Longueur dataset: {len(dataset)}")

doc_0 = dataset[0]
print(f"Type 1er document: {type(doc_0)}")
print(f"Clés d'un document: {doc_0.keys()}")

Type dataset: <class 'list'>
Longueur dataset: 200
Type 1er document: <class 'dict'>
Clés d'un document: dict_keys(['doc_id', 'date_publication', 'titre', 'auteur', 'auteur_genre', 'auteur_nom', 'auteur_prenom', 'genre_litteraire', 'texte', 'tokens', 'lemmes'])


In [7]:
for key, value in doc_0.items():
    print(f"{key:<20} {type(value)}    {str(value)[:100]}")

doc_id               <class 'str'>    HDU-00027
date_publication     <class 'int'>    1801
titre                <class 'str'>    Atala, ou les Amours de deux sauvages dans le désert
auteur               <class 'str'>    François-René de Chateaubriand
auteur_genre         <class 'str'>    masculin
auteur_nom           <class 'str'>    Chateaubriand
auteur_prenom        <class 'str'>    François-René de
genre_litteraire     <class 'str'>    autre
texte                <class 'str'>    Ans sur cette quatrième Edition'. -S quelque temps, il a paru de nouvelles critiques d'Atala. Je n'a
tokens               <class 'list'>    ['Ans', 'sur', 'cette', 'quatrième', 'Edition', "'", '.', '-S', 'quelque', 'temps', ',', 'il', 'a', 
lemmes               <class 'list'>    ['an', 'sur', 'ce', 'quatrième', 'edition', "'", '.', '-s', 'quelque', 'temps', ',', 'il', 'avoir', 


### Vocabulaire

Construire un dictionnaire du vocabulaire du corpus.

### Choix de l'unité élémentaire : tokens ou lemmes ?

In [8]:
for word_type in ["tokens", "lemmes"]:
    all_words = [word.lower()
                  for document in dataset
                  for word in document[word_type]
                  ]
    vocab = sorted(set(all_words))
    print(f"{word_type:<10}  Taille du vocabulaire: {len(vocab):,} words")

tokens      Taille du vocabulaire: 322,838 words
lemmes      Taille du vocabulaire: 285,868 words


Nous retenons les lemmes pour réduire la taille du vocabulaire

In [119]:
word_type = "lemmes" # ["lemmes", "tokens"]

In [122]:
all_words = [word.lower()
                  for document in dataset
                  for word in document[word_type]
                  ]
vocab = sorted(set(all_words))


print(f"Documents : {len(dataset):,}")
print(f"Taille du corpus : {len(all_words):,}")
print(f"Mots / Document : {int(len(all_words)/len(dataset)):,}")
print(f"Taille du vocabulaire : {len(vocab):,} ({word_type})")

Documents : 200
Taille du corpus : 19,719,818
Mots / Document : 98,599
Taille du vocabulaire : 285,868 (lemmes)


In [123]:
import pandas as pd
from collections import Counter

vocab_counter = Counter(all_words).most_common()

df = pd.DataFrame([
    {
        "Rang": i + 1,
        "Word": word,
        "Occurrences": count,
        "Fréquence (%)": 100 * count / len(all_words)
    }
    for i, (word, count) in enumerate(vocab_counter)
])

df

,Rang,Word,Occurrences,Fréquence (%)
0,1,",",1337995,6.785027
1,2,le,1252190,6.349906
2,3,de,1067224,5.411936
3,4,.,693278,3.515641
4,5,un,434497,2.203352
...,...,...,...,...
285863,285864,mous,1,0.000005
285864,285865,silenciesuement,1,0.000005
285865,285866,téqui,1,0.000005
285866,285867,1948,1,0.000005


In [124]:
word_lists = [
    [word.lower() for word in doc[word_type]]
    for doc in dataset
]
word_counts = Counter(all_words)

print(f"Vocabulaire : {len(vocab_counter):,}")
print(f"Mots apparaissant une seule fois : {sum(c[1] == 1 for c in vocab_counter):,}")
print(f"Mots apparaissant 5 fois ou plus : {sum(c[1] >= 5 for c in vocab_counter):,}")
print(f"Mots apparaissant 20 fois ou plus : {sum(c[1] >= 20 for c in vocab_counter):,}")
print(f"Mots apparaissant 100 fois ou plus : {sum(c[1] >= 100 for c in vocab_counter):,}")
print(f"Mots apparaissant 1 000 fois ou plus : {sum(c[1] >= 1000 for c in vocab_counter):,}")

Vocabulaire : 285,868
Mots apparaissant une seule fois : 189,014
Mots apparaissant 5 fois ou plus : 43,203
Mots apparaissant 20 fois ou plus : 20,367
Mots apparaissant 100 fois ou plus : 7,926
Mots apparaissant 1 000 fois ou plus : 1,528


In [ ]:
import numpy as np
from scipy.sparse import coo_matrix
from tqdm import tqdm

N = 10

word_to_id = {word: i for i, word in enumerate(vocab)}

rows = []
cols = []

for tokens in tqdm(word_lists):

    ids = np.fromiter(
        (word_to_id[w] for w in tokens if w in word_to_id),
        dtype=np.int32
    )

    # Generate context pairs for each distance
    for d in range(1, N + 1):

        if len(ids) <= d:
            continue

        left = ids[:-d]
        right = ids[d:]

        # Both directions: target -> context and context -> target
        rows.append(np.concatenate((left, right)))
        cols.append(np.concatenate((right, left)))

# Concatenate all pairs
rows = np.concatenate(rows)
cols = np.concatenate(cols)

# Build sparse matrix
cooc_matrix = coo_matrix(
    (
        np.ones(len(rows), dtype=np.int32),
        (rows, cols)
    ),
    shape=(len(vocab), len(vocab))
).tocsr()

print("Dimensions de la matrice :", cooc_matrix.shape)
print(f"Nombre d'entrées non nulles : {cooc_matrix.nnz:,} ({cooc_matrix.nnz/(cooc_matrix.shape[0]*cooc_matrix.shape[1]):.4%})")

100%|██████████| 200/200 [00:07<00:00, 25.78it/s]


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_most_similar(
    word,
    matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=10,
    min_count=1
):

    if word not in word_to_id:
        print(f"'{word}' is not in the vocabulary.")
        return

    word_id = word_to_id[word]

    # Compute similarity against all word vectors
    similarities = cosine_similarity(
        matrix[word_id],
        matrix
    ).flatten().tolist()

    # Rank candidates
    closest_ids = np.argsort(similarities)[::-1]

    results = []

    for idx in closest_ids:

        candidate = vocab[idx]

        # Exclude the query word
        if idx == word_id:
            continue

        # Apply frequency threshold
        if word_counts[candidate] < min_count:
            continue

        results.append((candidate, round(similarities[idx], 4)))

        if len(results) == top_n:
            break

    return results

In [ ]:
min_word_occurrences = 0

filtered_vocab = [
    word for word, count in word_counts.items()
    if count >= min_word_occurrences
]
print(f"Vocabulaire après filtrage: {len(filtered_vocab):,}")

In [ ]:
target_word = "mère"
most_similar = get_most_similar(
    target_word,
    cooc_matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=10,
    min_count=min_word_occurrences
)
print(f"Mot Cible  : {target_word}")
print(f"Contexte N : {N}")
print(f"-----------------")
print(pd.DataFrame(most_similar, columns=["Word", "Cos_Sim"]).to_string(index=False, header=False, justify=["right", "center"]))

In [ ]:
# Build a sparse matrix
rows = [pair[0] for pair in pair_counts]
cols = [pair[1] for pair in pair_counts]
data = list(pair_counts.values())

cooc_matrix = coo_matrix(
    (data, (rows, cols)),
    shape=(len(vocab), len(vocab))
).tocsr()

# L1 normalization: each non-zero row sums to 1
cooc_matrix_normalized = normalize(
    cooc_matrix,
    norm="l1",
    axis=1
)

In [ ]:
most_similar(
    "homme",
    cooc_matrix_normalized,
    word_to_id,
    vocab,
    top_n=10
)

In [ ]:
for word in ["chien", "chat", "maison", "voiture"]:
    print(word, word_counts[word])

In [ ]:
import numpy as np
from scipy.sparse import coo_matrix

def compute_ppmi(cooc_matrix):
    cooc = cooc_matrix.tocoo()

    total = cooc.data.sum()
    row_sums = np.asarray(cooc_matrix.sum(axis=1)).ravel()
    col_sums = np.asarray(cooc_matrix.sum(axis=0)).ravel()

    pmi = np.log(
        (cooc.data * total) /
        (row_sums[cooc.row] * col_sums[cooc.col])
    )

    ppmi = np.maximum(pmi, 0)

    return coo_matrix(
        (ppmi, (cooc.row, cooc.col)),
        shape=cooc_matrix.shape
    ).tocsr()

ppmi_matrix = compute_ppmi(cooc_matrix)

In [ ]:
most_similar(
    "femme",
    ppmi_matrix,
    word_to_id,
    vocab,
    top_n=10
)

In [ ]:
from sklearn.decomposition import TruncatedSVD

def compute_ppmi(cooc_matrix, alpha=0.75, shift=0.0):
    cooc = cooc_matrix.tocoo()

    total = cooc.data.sum()
    row_sums = np.asarray(cooc_matrix.sum(axis=1)).ravel()
    col_sums = np.asarray(cooc_matrix.sum(axis=0)).ravel() ** alpha

    p_wc = cooc.data / total
    p_w = row_sums[cooc.row] / total
    p_c = col_sums[cooc.col] / col_sums.sum()

    pmi = np.log(p_wc / (p_w * p_c))
    ppmi = np.maximum(pmi - shift, 0)   # shift = np.log(k) gives shifted PPMI

    result = coo_matrix((ppmi, (cooc.row, cooc.col)), shape=cooc_matrix.shape).tocsr()
    result.eliminate_zeros()
    return result

ppmi_matrix = compute_ppmi(cooc_matrix, alpha=0.75)

svd = TruncatedSVD(n_components=300, random_state=0)
us = svd.fit_transform(ppmi_matrix)              # U * S
embeddings = us / np.sqrt(svd.singular_values_)  # U * S^0.5

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def most_similar(word, matrix, word_to_id, vocab, top_n=10):
    if word not in word_to_id:
        print(f"'{word}' is not in the vocabulary.")
        return

    word_id = word_to_id[word]
    sims = cosine_similarity(matrix[word_id:word_id + 1], matrix).ravel()
    sims[word_id] = -np.inf                      # exclude the word itself

    top = np.argpartition(sims, -top_n)[-top_n:]  # ten best, unordered
    top = top[np.argsort(sims[top])[::-1]]        # sort only those ten

    return [(vocab[i], float(sims[i])) for i in top]

most_similar("femme", embeddings, word_to_id, vocab, top_n=10)

In [ ]:
from sklearn.preprocessing import normalize

emb_norm = normalize(embeddings)   # each row gets unit L2 norm
def analogy(a, b, c, top_n=5):
    # a is to b as c is to ?
    v = emb_norm[word_to_id[b]] - emb_norm[word_to_id[a]] + emb_norm[word_to_id[c]]
    sims = emb_norm @ (v / np.linalg.norm(v))
    for w in (a, b, c):
        sims[word_to_id[w]] = -np.inf
    top = np.argsort(sims)[::-1][:top_n]
    return [(vocab[i], float(sims[i])) for i in top]

print(analogy("homme", "femme", "roi"))    # reine should rank high
print(analogy("père", "mère", "fils"))     # fille

In [ ]:
token_dict = {key: {} for key in vocab}

In [ ]:
# from collections import Counter
# import pandas as pd
#
# X = pd.DataFrame(
#     [Counter(document["tokens"]) for document in dataset]
# ).fillna(0).astype("int32")
#
# X.index.name = "document"

In [ ]:
print(Counter(all_tokens).most_common(50))

#### Réduction du vocabulaire

In [ ]:
token_count = dict(Counter(all_tokens))
token_unique = set([token for token, count in token_count.items() if count == 1])

print(f"{len(token_unique):,} tokens n'aparaissent qu'une seule fois dans l'ensemble du corpus. ({len(token_unique):,} / {len(vocab):,} = {len(token_unique)/len(vocab):.2%})")

In [ ]:
filtered_vocab = set([token for token in vocab if token not in token_unique])
print(f"Taille du vocabulaire: {len(filtered_vocab):,} tokens")

Pour être utile à la classification, un token doit apparaître dans plusieurs documents. On cherche donc à savoir dans combien de document aparaissent chacun des tokens du vocabulaire.

In [ ]:
vocab_documents = {token: {"occurrence": 0} for token in filtered_vocab}
for document in tqdm(dataset):
    tokens = set(document['tokens'])
    for token in tokens:
        if token in filtered_vocab:
            vocab_documents[token]["occurrence"] += 1
dataset_size = len(dataset)
for token in tqdm(vocab_documents):
    vocab_documents[token]["frequency"] = vocab_documents[token]["occurrence"]/dataset_size

In [ ]:
min_documents_count = 5
max_documents_count = 300
# min_documents_ratio = 0.1
# max_documents_ratio = 0.8
min_token_length = 2

filtered_vocab = set([token for token in vocab_documents if vocab_documents[token]["occurrence"] >= min_documents_count
                                                    and vocab_documents[token]["occurrence"] <= max_documents_count
                                                    # and vocab_documents[token]["frequency"] >= min_documents_ratio
                                                    # and vocab_documents[token]["frequency"] <= max_documents_ratio
                                                    and len(token) >= min_token_length
                 ])
print(f"Taille du vocabulaire: {len(filtered_vocab):,} tokens")

In [ ]:
for document in tqdm(dataset):
    document["occurrence"] = dict(Counter([token for token in document["tokens"] if token in filtered_vocab]).most_common())
    filtered_tokens_count = sum(list(document["occurrence"].values()))
    document["frequence"] = {token:occurrence/filtered_tokens_count for token, occurrence in document["occurrence"].items()}

In [ ]:
unique_labels = sorted(set(document[label_column] for document in dataset))

label2id = {
    genre: i
    for i, genre in enumerate(unique_labels)
}

id2label = {
    i: genre
    for genre, i in label2id.items()
}

print(label2id)
print(id2label)

In [ ]:
import pandas as pd

X_matrix = pd.DataFrame([document["frequence"] for document in dataset]).fillna(0)
X_matrix.head()
X_matrix.shape

In [ ]:
X = X_matrix
y = pd.Series([
    label2id[document[label_column]]
    for document in dataset
])

groups = pd.Series([
    document["auteur"]
    for document in dataset
])

### Baseline Model - Naive Bayes / All vocabulary / its a NLP tp not ML

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import LeaveOneGroupOut
from tqdm import tqdm
import pandas as pd

X_matrix = pd.DataFrame(
    [document["frequence"] for document in dataset]
).fillna(0)

X = X_matrix

y = pd.Series([
    label2id[document[label_column]]
    for document in dataset
])

groups = pd.Series([
    document["auteur"]
    for document in dataset
])

logo = LeaveOneGroupOut()

results = []

for train_idx, test_idx in tqdm(
    logo.split(X, y, groups),
    total=groups.nunique(),
    desc="Leave-one-author-out"
):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    authors = groups.iloc[test_idx]

    model = RandomForestClassifier(
        max_depth=50,
        max_features="log2",
        min_samples_leaf=2,
        min_samples_split=2,
        n_estimators=400,
        n_jobs=4,
        random_state=42
    )
    # from sklearn.svm import LinearSVC
    #
    # model = LinearSVC(
    #     C=1.0,
    #     class_weight="balanced"
    # )
    # from sklearn.linear_model import LogisticRegression
    # model = LogisticRegression(
    #     C=1,
    #     max_iter=3000,
    #     class_weight="balanced"
    # )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    for author, gold, pred in zip(authors, y_test, y_pred):
        results.append({
            "auteur": author,
            "gold_label": gold,
            "predicted_label": pred
        })

results_df = pd.DataFrame(results)

In [ ]:
accuracy = accuracy_score(
    results_df["gold_label"],
    results_df["predicted_label"]
)

print(f"Accuracy: {accuracy:.3f}")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report

texts = [" ".join(d["lemmes"]) for d in dataset]
y = [d[label_column] for d in dataset]
groups = [d["auteur"] for d in dataset]

models = {
    "majority class": DummyClassifier(strategy="most_frequent"),
    "naive bayes": MultinomialNB(),
    "logistic regression": LogisticRegression(max_iter=3000),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

for name, clf in models.items():
    pipe = make_pipeline(CountVectorizer(min_df=5, max_df=0.5), clf)
    y_pred = cross_val_predict(pipe, texts, y, groups=groups, cv=cv)
    print(f"{name:<22}{accuracy_score(y, y_pred):.3f}")

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score

def identity(doc):
    return doc

inputs = {
    "tokens": [d["tokens"] for d in dataset],
    "lemmes": [d["lemmes"] for d in dataset],
}

weightings = {
    "binary": lambda: CountVectorizer(analyzer=identity, binary=True, min_df=5, max_df=0.5),
    "count":  lambda: CountVectorizer(analyzer=identity, min_df=5, max_df=0.5),
    "tf":     lambda: TfidfVectorizer(analyzer=identity, use_idf=False, norm="l1", min_df=5, max_df=0.5),
    "tfidf":  lambda: TfidfVectorizer(analyzer=identity, sublinear_tf=True, min_df=5, max_df=0.5),
}

models = {
    "naive bayes": lambda: MultinomialNB(),
    "logreg":      lambda: LogisticRegression(max_iter=3000),
    "linear svm":  lambda: LinearSVC(C=1.0),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
splits = list(cv.split(inputs["lemmes"], y, groups))   # computed once, reused everywhere

i = 0
rows = []
for inp_name, docs in inputs.items():
    for w_name, make_vec in weightings.items():
        for m_name, make_model in models.items():
            pipe = make_pipeline(make_vec(), make_model())
            y_pred = cross_val_predict(pipe, docs, y, cv=splits, n_jobs=-1)

            config = {
                "input": inp_name, "weighting": w_name, "model": m_name,
                "acc": accuracy_score(y, y_pred),
                "macro_f1": f1_score(y, y_pred, average="macro"),
            }
            rows.append(config)
            print(config)
            i += 1

results = pd.DataFrame(rows)
results.pivot_table(index=["input", "weighting"], columns="model", values="acc").round(3)